# BioAI Hub — Layer 5: Transfer Learning con EfficientNet-B3

En layers anteriores entrené BioAICNN desde cero sobre Chest X-Ray Kaggle (2 clases, ~6k imágenes). Acá hago fine-tuning de EfficientNet-B3 pre-entrenado en ImageNet sobre NIH ChestX-ray14 (14 patologías, 112k imágenes).

La diferencia fundamental: en vez de inicializar con pesos aleatorios, parto de un modelo que ya aprendió a detectar bordes, texturas y formas en millones de imágenes. Solo tengo que enseñarle a aplicar ese conocimiento a radiografías.

## 0. Setup

In [ ]:
import os, pickle
import torch
import torch.nn as nn
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/bioai-colab'
NIH_DIR  = os.path.join(WORK_DIR, 'nih-data')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cargar splits y pesos guardados en Layer 4
with open(os.path.join(WORK_DIR, 'nih_splits.pkl'), 'rb') as f:
    estado = pickle.load(f)

df_train     = estado['df_train']
df_val       = estado['df_val']
df_test      = estado['df_test']
pos_weights  = estado['pos_weights']
PATOLOGIAS   = estado['patologias']

print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"Patologías: {PATOLOGIAS}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class NIHChestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Image Index'])
        image    = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        labels_str = row['Finding Labels']
        label_vec  = torch.zeros(len(PATOLOGIAS), dtype=torch.float32)
        if labels_str != 'No Finding':
            for pat in labels_str.split('|'):
                if pat in PATOLOGIAS:
                    label_vec[PATOLOGIAS.index(pat)] = 1.0

        return image, label_vec


transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

img_dir = os.path.join(NIH_DIR, 'images')

train_ds = NIHChestDataset(df_train, img_dir, transform=transform_train)
val_ds   = NIHChestDataset(df_val,   img_dir, transform=transform_eval)
test_ds  = NIHChestDataset(df_test,  img_dir, transform=transform_eval)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader:   {len(val_loader)} batches")

## 1. Modelo — EfficientNet-B3

EfficientNet escala ancho, profundidad y resolución de forma balanceada. La variante B3 tiene ~12M de parámetros — lo suficientemente grande para capturar patrones complejos en radiografías sin ser prohibitivo en tiempo de entrenamiento.

Reemplazo la última capa del clasificador por una de 14 salidas, una por patología. No uso softmax — `BCEWithLogitsLoss` aplica sigmoid internamente por patología, lo que permite que múltiples clases sean positivas al mismo tiempo.

In [ ]:
import torchvision.models as models

def construir_efficientnet(num_classes=14, freeze_backbone=True):
    model = models.efficientnet_b3(weights='IMAGENET1K_V1')

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Reemplazar clasificador final
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model.to(device)


model = construir_efficientnet(freeze_backbone=True)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parámetros totales:      {total_params:,}")
print(f"Parámetros entrenables:  {trainable_params:,}  (solo el clasificador)")

## 2. Fine-tuning en dos fases

**Fase 1** — backbone congelado, solo entreno el clasificador. El backbone de EfficientNet ya sabe extraer features. Si lo descongelara desde el principio con lr alto, los pesos pre-entrenados se destruirían antes de que el clasificador se estabilice.

**Fase 2** — desglaceo todo y entreno con lr mucho más bajo. Ahora el backbone se ajusta sutilmente para features específicas de radiografías sin perder el conocimiento de ImageNet.

In [ ]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_labels, all_probs = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = model(images)
            loss    = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_labels.append(labels.cpu().numpy())
            all_probs.append(torch.sigmoid(outputs).detach().cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_probs  = np.vstack(all_probs)

    # AUC-ROC promedio sobre patologías con al menos 2 casos positivos
    aucs = []
    for i in range(len(PATOLOGIAS)):
        if all_labels[:, i].sum() >= 2:
            aucs.append(roc_auc_score(all_labels[:, i], all_probs[:, i]))

    mean_auc = np.mean(aucs) if aucs else 0.0
    avg_loss = total_loss / len(loader.dataset)

    return avg_loss, mean_auc, all_labels, all_probs


def guardar_checkpoint(model, optimizer, epoch, val_loss, val_auc, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        'val_auc': val_auc,
    }, path)

print("Utilidades definidas")

In [ ]:
# Fase 1: solo el clasificador, LR alto, pocas epochs
CHECKPOINT_F1 = os.path.join(WORK_DIR, 'efficientnet_fase1_best.pth')

optimizer_f1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)

EPOCHS_F1   = 3
best_auc_f1 = 0.0

print("=== Fase 1: backbone congelado ===")
for epoch in range(1, EPOCHS_F1 + 1):
    train_loss, train_auc, _, _ = run_epoch(model, train_loader, optimizer=optimizer_f1)
    val_loss,   val_auc,   _, _ = run_epoch(model, val_loader)

    mejoro = val_auc > best_auc_f1
    if mejoro:
        best_auc_f1 = val_auc
        guardar_checkpoint(model, optimizer_f1, epoch, val_loss, val_auc, CHECKPOINT_F1)

    print(f"Epoch {epoch}/{EPOCHS_F1}  "
          f"train_loss={train_loss:.4f}  train_auc={train_auc:.3f}  "
          f"val_loss={val_loss:.4f}  val_auc={val_auc:.3f}"
          f"{'  ← guardado' if mejoro else ''}")

print(f"\nMejor AUC Fase 1: {best_auc_f1:.4f}")

In [ ]:
# Fase 2: descongelar todo, LR bajo
CHECKPOINT_F2 = os.path.join(WORK_DIR, 'efficientnet_fase2_best.pth')

# Cargar mejor checkpoint de Fase 1
ckpt = torch.load(CHECKPOINT_F1, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

# Descongelar backbone
for param in model.parameters():
    param.requires_grad = True

optimizer_f2 = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_f2, mode='max', factor=0.5, patience=2
)

EPOCHS_F2   = 5
best_auc_f2 = 0.0

print("=== Fase 2: backbone descongelado ===")
for epoch in range(1, EPOCHS_F2 + 1):
    train_loss, train_auc, _, _ = run_epoch(model, train_loader, optimizer=optimizer_f2)
    val_loss,   val_auc,   _, _ = run_epoch(model, val_loader)

    scheduler.step(val_auc)

    mejoro = val_auc > best_auc_f2
    if mejoro:
        best_auc_f2 = val_auc
        guardar_checkpoint(model, optimizer_f2, epoch, val_loss, val_auc, CHECKPOINT_F2)

    print(f"Epoch {epoch}/{EPOCHS_F2}  "
          f"train_loss={train_loss:.4f}  train_auc={train_auc:.3f}  "
          f"val_loss={val_loss:.4f}  val_auc={val_auc:.3f}"
          f"{'  ← guardado' if mejoro else ''}")

print(f"\nMejor AUC Fase 2: {best_auc_f2:.4f}")

## 3. Evaluación en test set

In [ ]:
import matplotlib.pyplot as plt

# Cargar mejor checkpoint de Fase 2
ckpt_f2 = torch.load(CHECKPOINT_F2, map_location=device)
model.load_state_dict(ckpt_f2['model_state_dict'])
print(f"Checkpoint cargado — epoch {ckpt_f2['epoch']}, val_auc={ckpt_f2['val_auc']:.4f}")

_, _, test_labels, test_probs = run_epoch(model, test_loader)

# AUC-ROC por patología
print("\n--- AUC-ROC por patología (test set) ---")
aucs_por_pat = {}
for i, pat in enumerate(PATOLOGIAS):
    n_pos = int(test_labels[:, i].sum())
    if n_pos >= 2:
        auc = roc_auc_score(test_labels[:, i], test_probs[:, i])
        aucs_por_pat[pat] = auc
        print(f"  {pat:25s}  AUC={auc:.3f}  ({n_pos} casos positivos)")
    else:
        print(f"  {pat:25s}  — (insuficientes casos positivos)")

mean_auc = np.mean(list(aucs_por_pat.values()))
print(f"\nAUC-ROC promedio: {mean_auc:.4f}")

In [ ]:
# Gráfico de AUC por patología
pats  = list(aucs_por_pat.keys())
aucs  = list(aucs_por_pat.values())
colores = ['#2ecc71' if a >= 0.80 else '#e67e22' if a >= 0.70 else '#e74c3c' for a in aucs]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(pats, aucs, color=colores, alpha=0.85)
ax.axhline(y=mean_auc, color='navy', linestyle='--', alpha=0.6, label=f'Promedio ({mean_auc:.3f})')
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4, label='Azar (0.5)')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('AUC-ROC')
ax.set_title('EfficientNet-B3 — AUC-ROC por patología (NIH ChestX-ray14)')
ax.set_xticklabels(pats, rotation=40, ha='right')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'efficientnet_auc_por_patologia.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. Comparativa BioAICNN vs EfficientNet-B3

No es una comparación directa — son datasets y tareas distintas. BioAICNN resuelve clasificación binaria sobre ~6k imágenes. EfficientNet resuelve multi-label sobre 112k. Pero sirve para mostrar la progresión del proyecto.

In [ ]:
print("=" * 60)
print(f"{'':30s} {'BioAICNN':>12} {'EfficientNet-B3':>15}")
print("=" * 60)
print(f"{'Dataset':30s} {'Kaggle':>12} {'NIH':>15}")
print(f"{'Imágenes':30s} {'5,800':>12} {'112,120':>15}")
print(f"{'Clases':30s} {'2':>12} {'14':>15}")
print(f"{'Tipo de clasificación':30s} {'binaria':>12} {'multi-label':>15}")
print(f"{'Parámetros':30s} {'~500k':>12} {'~12M':>15}")
print(f"{'AUC-ROC':30s} {'0.942':>12} {mean_auc:>15.3f}")
print(f"{'Pesos iniciales':30s} {'aleatorios':>12} {'ImageNet':>15}")
print("=" * 60)
print()
print("Nota: AUC-ROC más bajo en NIH es esperado — 14 patologías con")
print("clases muy desbalanceadas es un problema mucho más difícil.")
print("Los papers de referencia reportan 0.74–0.84 por patología.")

## 5. Exportar modelo

In [ ]:
# Guardar pesos finales
torch.save(
    model.state_dict(),
    os.path.join(WORK_DIR, 'efficientnet_b3_nih_final.pth')
)

# Exportar a ONNX (portable, corre en cualquier framework)
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model,
    dummy_input,
    os.path.join(WORK_DIR, 'efficientnet_b3_nih.onnx'),
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=17,
)

print("Archivos exportados:")
for archivo in sorted(os.listdir(WORK_DIR)):
    path = os.path.join(WORK_DIR, archivo)
    if os.path.isfile(path):
        print(f"  {archivo:50s} {os.path.getsize(path)/1e6:.1f} MB")
    else:
        print(f"  {archivo}/")